In [ ]:
import os
import numpy as np
from sklearn.datasets import load_svmlight_file

# Define paths to our data
DATA_DIR = "../data/Fold1"
TRAIN_FILE = os.path.join(DATA_DIR, "train.txt")

if not os.path.exists(TRAIN_FILE):
    raise FileNotFoundError(f"Missing data at {TRAIN_FILE}. Ensure Fold1 is in the data directory.")

print("Initiating high-performance data ingestion pipeline...")
print("Loading 1.2 million query-document pairs into memory. This may take a moment...\n")

# load_svmlight_file parses the libsvm format and extracts the query_id natively
X_train, y_train, qid_train = load_svmlight_file(TRAIN_FILE, query_id=True)

print("✅ INGESTION COMPLETE")
print("-" * 30)
print(f"Feature Matrix (X) Shape:  {X_train.shape} -> (Rows, 136 Features)")
print(f"Labels Array (y) Shape:    {y_train.shape} -> (Relevance scores 0 to 4)")
print(f"Query IDs (qid) Shape:     {qid_train.shape} -> (Grouping metadata)")
print("-" * 30)

# Let's inspect the memory footprint of our sparse matrix
memory_mb = X_train.data.nbytes / (1024 * 1024)
print(f"Sparse Matrix RAM Usage: ~{memory_mb:.2f} MB")

In [4]:
import lightgbm as lgb

# 1. Ingest Validation Data (Crucial for scoring our search engine)
VALI_FILE = os.path.join(DATA_DIR, "vali.txt")
print("Ingesting validation data for NDCG scoring...")
X_vali, y_vali, qid_vali = load_svmlight_file(VALI_FILE, query_id=True)

# 2. Calculate Group Sizes (The core requirement for Learning-to-Rank)
# LightGBM requires an array of counts per query, not the raw query IDs
_, group_train = np.unique(qid_train, return_counts=True)
_, group_vali = np.unique(qid_vali, return_counts=True)

print(f"Training Queries: {len(group_train)} | Validation Queries: {len(group_vali)}")

# 3. Construct the LightGBM Engine Datasets
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
vali_data = lgb.Dataset(X_vali, label=y_vali, group=group_vali, reference=train_data)
print("✅ Engine primed for LambdaMART compilation.\n")

# 4. Define the LambdaMART Architecture
params = {
    'objective': 'lambdarank',       # Pairwise ranking objective
    'metric': 'ndcg',                # Normalized Discounted Cumulative Gain
    'ndcg_eval_at': [5, 10],         # Optimize for the Top 5 and Top 10 search results
    'learning_rate': 0.05,
    'num_leaves': 31,                # Standard tree complexity
    'min_data_in_leaf': 50,
    'verbose': -1                    # Suppress C++ warnings
}

print("Booting LambdaMART training sequence...")

# 5. Train the Model
model = lgb.train(
    params,
    train_data,
    num_boost_round=100,             # Number of trees to build
    valid_sets=[train_data, vali_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=15),
        lgb.log_evaluation(period=10)
    ]
)

Ingesting validation data for NDCG scoring...
Training Queries: 6000 | Validation Queries: 2000
✅ Engine primed for LambdaMART compilation.

Booting LambdaMART training sequence...
Training until validation scores don't improve for 15 rounds
[10]	train's ndcg@5: 0.430705	train's ndcg@10: 0.449737	valid's ndcg@5: 0.43685	valid's ndcg@10: 0.455165
[20]	train's ndcg@5: 0.44313	train's ndcg@10: 0.463372	valid's ndcg@5: 0.446942	valid's ndcg@10: 0.466825
[30]	train's ndcg@5: 0.452673	train's ndcg@10: 0.471985	valid's ndcg@5: 0.454625	valid's ndcg@10: 0.473867
[40]	train's ndcg@5: 0.461176	train's ndcg@10: 0.479719	valid's ndcg@5: 0.460203	valid's ndcg@10: 0.479306
[50]	train's ndcg@5: 0.469527	train's ndcg@10: 0.486638	valid's ndcg@5: 0.466719	valid's ndcg@10: 0.484966
[60]	train's ndcg@5: 0.47489	train's ndcg@10: 0.49152	valid's ndcg@5: 0.470271	valid's ndcg@10: 0.489785
[70]	train's ndcg@5: 0.480367	train's ndcg@10: 0.495955	valid's ndcg@5: 0.475071	valid's ndcg@10: 0.493078
[80]	train's 